# Ecowitt — schema explorer

**What is in the database, and what shape is it.** Every object is discovered
from the catalog at runtime, so anything added later is automatically in scope.

This notebook is for poking around. Two siblings do the other jobs:

| notebook | question |
|---|---|
| `explore.ipynb` (this one) | what exists, and how is it built |
| `explore_daily.ipynb` | is yesterday's data sound, and what did the weather do |
| `audit.ipynb` | does the whole record hold up, all-time |

**No prerequisite.** The tunnel is opened automatically if one is not already
running, and left alone if another notebook is using it. Postgres binds to
localhost on the VM and nothing accepts inbound connections, so a forwarded
local port is the only way in — that part cannot be automated away, only the
opening of it.

Read-only throughout: the `ecowitt_ro` role is refused writes twice over, by
grant and by a forced read-only transaction default.

In [ ]:
# Preflight: make this notebook work whatever kernel it lands on.
#
# VS Code writes the selected kernel back INTO this file, so pinning a
# kernelspec here loses to whatever was last used. This machine also has
# several Pythons (Anaconda, other project venvs). Rather than fight that,
# fall back to the project venv's site-packages when the current interpreter
# lacks the dependencies -- safe only when the Python versions match exactly,
# because compiled extensions like psycopg are ABI-specific.
import sys
from importlib.util import find_spec
from pathlib import Path

NEEDED = ("psycopg", "sqlalchemy", "pandas", "matplotlib")


def _repo_venv_site_packages():
    """Walk up from the notebook looking for a sibling .venv."""
    for base in (Path.cwd(), *Path.cwd().parents):
        version = f"python{sys.version_info.major}.{sys.version_info.minor}"
        sp = base / ".venv" / "lib" / version / "site-packages"
        if sp.is_dir():
            return sp
    return None


missing = [m for m in NEEDED if find_spec(m) is None]
if missing:
    sp = _repo_venv_site_packages()
    if sp and str(sp) not in sys.path:
        sys.path.insert(0, str(sp))
        missing = [m for m in NEEDED if find_spec(m) is None]
        if not missing:
            print(f"NOTE: running on {sys.executable}")
            print(f"      which lacks {', '.join(NEEDED)}, so this notebook added")
            print(f"      {sp}")
            print("      to sys.path as a fallback. It works because both interpreters are")
            micro = sys.version_info.micro
            print(f"      Python {sys.version_info.major}."
                  f"{sys.version_info.minor}.{micro}.")
            print("      Cleaner: select the 'Python (ecowitt)' kernel (VS Code: kernel name,")
            print("      top-right of the notebook).")

if missing:
    raise SystemExit(
        f"Wrong kernel, and the fallback could not help.\n\n"
        f"  running on : {sys.executable}\n"
        f"  missing    : {', '.join(missing)}\n\n"
        "Select the 'Python (ecowitt)' kernel.\n"
        "  VS Code    : click the kernel name in the notebook's TOP-RIGHT corner\n"
        "               -> Select Another Kernel... -> Jupyter Kernel...\n"
        "  JupyterLab : Kernel > Change Kernel\n\n"
        "If it is not offered, register it once from the repo root:\n"
        "    .venv/bin/python -m ipykernel install --user --name ecowitt \\\n"
        "        --display-name 'Python (ecowitt)'\n\n"
        "If only matplotlib is missing, the venv predates the charting section:\n"
        "    .venv/bin/python -m pip install -e '.[notebook]'"
    )

print(f"kernel OK: {sys.executable}")


In [ ]:
# Shared plumbing lives in notebooks/ecowitt_nb.py -- tunnel, connection,
# palette, fetching, metric metadata, charts. It is imported rather than
# pasted: the same cell copied into two notebooks had to be bug-fixed twice.
import importlib
import pathlib
import sys

import pandas as pd
from sqlalchemy import text

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / 'notebooks' / 'ecowitt_nb.py').is_file():
        sys.path.insert(0, str(_base / 'notebooks'))
        break
    if (_base / 'ecowitt_nb.py').is_file():
        sys.path.insert(0, str(_base))
        break
else:
    raise SystemExit('cannot find notebooks/ecowitt_nb.py above ' + str(pathlib.Path.cwd()))

import ecowitt_nb as nb  # noqa: E402  -- must follow the sys.path insert above

importlib.reload(nb)          # pick up edits to the module without a kernel restart

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 250)
nb.style()

conn = nb.ensure_db()
print(pd.read_sql(text('select current_user, current_database(), now()'), conn)
        .to_string(index=False))


## 1. What objects exist

Discovered from `pg_class`, not hardcoded. Sizes are total relation size, so
they include indexes and TOAST.

In [ ]:
conn = nb.ensure_db()

OBJECTS_SQL = '''
SELECT c.relname                       AS object_name,
       CASE c.relkind WHEN 'r' THEN 'table'
                      WHEN 'p' THEN 'partitioned table'
                      WHEN 'v' THEN 'view'
                      WHEN 'm' THEN 'materialized view'
       END                             AS kind,
       pg_size_pretty(pg_total_relation_size(c.oid)) AS size,
       obj_description(c.oid)          AS comment
  FROM pg_class c
  JOIN pg_namespace n ON n.oid = c.relnamespace
 WHERE n.nspname = 'public'
   AND c.relkind IN ('r', 'p', 'v', 'm')
 ORDER BY c.relkind, c.relname
'''

objects = pd.read_sql(text(OBJECTS_SQL), conn)
print(f'{len(objects)} objects discovered\n')
objects

## 2. How much is in them

Counted per object rather than read from `pg_class.reltuples`, which is only a
planner estimate and can be stale or plain wrong on a young table.

In [ ]:
counts = []
for row in objects.itertuples():
    try:
        n = conn.execute(text(f'SELECT count(*) FROM public."{row.object_name}"')).scalar()
        counts.append({'object_name': row.object_name, 'kind': row.kind,
                       'rows': n, 'error': None})
    except Exception as exc:
        conn.rollback()
        counts.append({'object_name': row.object_name, 'kind': row.kind,
                       'rows': None, 'error': str(exc).splitlines()[0][:80]})

pd.DataFrame(counts)

## 3. What the rows look like

A failure on one object is reported and skipped rather than aborting the batch —
an empty table and a broken view should be distinguishable at a glance.

In [ ]:
for row in objects.itertuples():
    print('=' * 100)
    print(f'{row.object_name}  ({row.kind})')
    print('=' * 100)
    try:
        df = pd.read_sql(
            text(f'SELECT * FROM public."{row.object_name}" LIMIT 5'), conn)
    except Exception as exc:
        conn.rollback()
        print(f'  ERROR: {str(exc).splitlines()[0][:120]}\n')
        continue
    display(df if not df.empty else '(no rows)')
    print()

## 4. How the schema enforces its own rules

This is the part worth actually reading. The schema is doing work that would
otherwise be a code review promise: five of the project's rules are made
**impossible to violate** here rather than merely detectable later.

| Mechanism | What it makes impossible |
|---|---|
| trigger on `raw_payload` | mutating a landed payload — raw is append-only |
| composite FK `(metric, unit)` | a unit changing underneath stored history |
| CHECK on `request_url` | a credential landing in the database |
| PK `(station_id, ts_utc, metric)` | a duplicate natural key |
| `source` enum | an observed row and a resampled one being confused

Choosing self-managed Postgres over BigQuery cost about $44/yr and bought
exactly this: BigQuery has no CHECK constraints, no triggers and no foreign
keys, so every row below would have become a thing to hope the code got right.

In [ ]:
CONSTRAINTS_SQL = '''
SELECT rel.relname AS table_name,
       con.conname AS constraint_name,
       CASE con.contype WHEN 'p' THEN 'primary key'
                        WHEN 'f' THEN 'foreign key'
                        WHEN 'c' THEN 'check'
                        WHEN 'u' THEN 'unique'
                        ELSE con.contype::text END AS kind,
       pg_get_constraintdef(con.oid)               AS definition
  FROM pg_constraint con
  JOIN pg_class rel      ON rel.oid = con.conrelid
  JOIN pg_namespace nsp  ON nsp.oid = rel.relnamespace
 WHERE nsp.nspname = 'public'
 ORDER BY rel.relname, con.contype DESC, con.conname
'''

constraints = pd.read_sql(text(CONSTRAINTS_SQL), conn)
print(f'{len(constraints)} constraints across {constraints.table_name.nunique()} tables')
print(constraints.kind.value_counts().to_string(), '\n')
display(constraints)

In [ ]:
TRIGGERS_SQL = '''
SELECT c.relname AS table_name, t.tgname AS trigger_name,
       pg_get_triggerdef(t.oid) AS definition
  FROM pg_trigger t
  JOIN pg_class c ON c.oid = t.tgrelid
  JOIN pg_namespace n ON n.oid = c.relnamespace
 WHERE NOT t.tgisinternal AND n.nspname = 'public'
 ORDER BY c.relname, t.tgname
'''

triggers = pd.read_sql(text(TRIGGERS_SQL), conn)
print(f'{len(triggers)} user triggers\n')
display(triggers)

INDEXES_SQL = '''
SELECT tablename AS table_name, indexname AS index_name, indexdef AS definition
  FROM pg_indexes WHERE schemaname = 'public'
 ORDER BY tablename, indexname
'''
indexes = pd.read_sql(text(INDEXES_SQL), conn)
print(f'\n{len(indexes)} indexes\n')
display(indexes)

## 5. Columns, per table

`observation` is long/tall — one row per (station, timestamp, metric) — rather
than wide. The reason is auditability: `change_log` records changes at field
level, so the table's key and the audit key are the same tuple. Adding a sensor
is then two inserts rather than a migration.

In [ ]:
COLUMNS_SQL = '''
SELECT table_name, ordinal_position AS pos, column_name, data_type,
       is_nullable, column_default
  FROM information_schema.columns
 WHERE table_schema = 'public'
 ORDER BY table_name, ordinal_position
'''

columns = pd.read_sql(text(COLUMNS_SQL), conn)
for table, group in columns.groupby('table_name'):
    print(f'--- {table} ({len(group)} columns)')
    display(group.drop(columns='table_name').set_index('pos'))

## 6. The metric catalog

The catalog is what makes a single aggregation rule impossible to apply blindly.
Every metric carries its own `resample_rule`, because one rule cannot serve all
of them — averaging a wind direction can point due south for a north wind, and
averaging a running total means nothing at all.

In [ ]:
catalog = pd.read_sql(text(
    'SELECT metric, kind, canonical_unit, resample_rule, resamplable '
    'FROM metric_catalog ORDER BY metric'), conn)

print(f'{len(catalog)} metrics')
print('\nby kind:')
print(catalog.kind.value_counts().to_string())
print('\nby rule:')
print(catalog.resample_rule.value_counts().to_string())
print('\nunits (exact bytes — never normalised, a º/° rewrite would corrupt history):')
print(catalog.canonical_unit.value_counts(dropna=False).to_string())
display(catalog)

## 7. What is actually stored, and over what span

A quick orientation before poking at anything specific: how far back the record
goes, and how complete it is.

In [ ]:
SPAN_SQL = '''
SELECT min(ts_utc) AS first_obs, max(ts_utc) AS last_obs,
       count(*) AS rows, count(DISTINCT ts_utc) AS timestamps,
       count(DISTINCT metric) AS metrics
  FROM observation
'''
span = pd.read_sql(text(SPAN_SQL), conn).iloc[0]
first_local = pd.Timestamp(span.first_obs).tz_convert(nb.DISPLAY_TZ)
last_local  = pd.Timestamp(span.last_obs).tz_convert(nb.DISPLAY_TZ)
covered = (pd.Timestamp(span.last_obs) - pd.Timestamp(span.first_obs)) / pd.Timedelta(minutes=5) + 1

print(f'first observation : {first_local:%Y-%m-%d %H:%M %Z}')
print(f'last observation  : {last_local:%Y-%m-%d %H:%M %Z}')
print(f'rows              : {span.rows:,}')
print(f'timestamps        : {span.timestamps:,} of {covered:,.0f} possible '
      f'5-minute slots ({100 * span.timestamps / covered:.1f}%)')
print(f'distinct metrics  : {span.metrics}')
print('\nNote the station has two eras: a 65-minute burst on 2026-07-23 with base')
print('sensors only, then nothing until 2026-08-01 when the extra channels were')
print('paired. Timestamps therefore legitimately carry 27 or 39 metrics.')

In [ ]:
PER_METRIC_SQL = '''
SELECT metric, count(*) AS rows,
       min(ts_utc) AS first_seen, max(ts_utc) AS last_seen,
       count(*) FILTER (WHERE value_num IS NULL) AS unparsed
  FROM observation GROUP BY metric ORDER BY metric
'''
per_metric = pd.read_sql(text(PER_METRIC_SQL), conn)
print('`unparsed` counts values kept as text because they would not cast — '
      'kept, never dropped.\n')
display(per_metric)

## 8. Poke around

Free-form space. `conn` is a live read-only connection; `nb.ensure_db()` gets it
back if a tunnel drops. Reads that mutate are refused by the role, so nothing
here can damage the record.

In [ ]:
# Example: the last hour of outdoor readings, straight from the long table.
pd.read_sql(text('''
    SELECT ts_utc, metric, value_text, unit
      FROM observation
     WHERE metric LIKE 'outdoor.%'
       AND ts_utc > now() - interval '1 hour'
     ORDER BY ts_utc DESC, metric
     LIMIT 40'''), conn)

In [ ]:
nb.shutdown()